## Hybrid Bergman and LSTM residual-corrector

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
# Single-cell notebook code: Hybrid Bergman minimal model + LSTM residual corrector
# Change simple params at the top to scale/adapt.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# PARAMETERS (easy to change)
DATA_PATH = "../data/OhioT1DM.csv"
LOOKBACK = 12            # number of 5-min timesteps used as input (12 -> 1 hour)
PRED_STEP = 6            # predict 6 timesteps ahead = 30 minutes
STEPS_AHEAD = 8          # repeat 8 times to reach 4 hours (8*30min)
EPOCHS = 3
BATCH_SIZE = 128
SUBJECTS_TO_PLOT = 3
DT = 5/60.0  # hours per timestep

# simple Bergman-like discrete simulator (very minimal)
def bergman_predict(G0, insulin_series, carb_series, basal_series, dt=5.0, steps=6,
                   p1=0.02, p2=0.02, p3=1.0, Gb=100.0):
    # G in mg/dL, insulin in units, carbs in grams
    G = float(G0)
    X = 0.0
    outputs = []
    for t in range(steps):
        I = insulin_series[t] if t < len(insulin_series) else 0.0
        C = carb_series[t] if t < len(carb_series) else 0.0
        B = basal_series[t] if t < len(basal_series) else basal_series[-1] if len(basal_series)>0 else 0.0
        # simplistic insulin action: insulin increases X
        dX = -p2*X + p3*(I)
        X = X + (dX * (dt/60.0))
        # carb absorption -> simple immediate glucose influx proportional to carbs (very simplified)
        carb_effect = 0.1 * C
        # glucose dynamics
        dG = -p1*(G - Gb) - X*G/100.0 + carb_effect
        G = G + dG * (dt/60.0)
        outputs.append(G)
    return outputs

# Load data
df = pd.read_csv(DATA_PATH, parse_dates=["date"])  # expects the path exists
# basic preprocessing
df = df.sort_values(["id","date"]).reset_index(drop=True)
# fill or interpolate small gaps
df['CGM'] = df['CGM'].interpolate().ffill()

subjects = df['id'].unique()[:SUBJECTS_TO_PLOT]

fig, axes = plt.subplots(SUBJECTS_TO_PLOT, 1, figsize=(10, 4*SUBJECTS_TO_PLOT))
if SUBJECTS_TO_PLOT == 1:
    axes = [axes]

for si, sid in enumerate(subjects):
    sub = df[df['id']==sid].copy().reset_index(drop=True)
    # construct arrays
    cgm = sub['CGM'].values
    carbs = sub['carbs'].fillna(0).values
    bolus = sub['bolus'].fillna(0).values
    basal = sub['basal'].fillna(0).values
    # build regression dataset
    Xs = []
    ys = []
    for i in range(LOOKBACK, len(sub)-PRED_STEP):
        # input window indices [i-LOOKBACK, i)
        in_slice = slice(i-LOOKBACK, i)
        seq_cgm = cgm[in_slice]
        seq_carbs = carbs[in_slice]
        seq_bolus = bolus[in_slice]
        seq_basal = basal[in_slice]
        # Bergman baseline: simulate from time i using small future input windows
        future_insulin = bolus[i:i+PRED_STEP].tolist()
        future_carbs = carbs[i:i+PRED_STEP].tolist()
        future_basal = basal[i:i+PRED_STEP].tolist()
        G0 = cgm[i-1]
        berg = bergman_predict(G0, future_insulin, future_carbs, future_basal, dt=5.0, steps=PRED_STEP)
        berg_30 = berg[-1]
        y = cgm[i+PRED_STEP-1] - berg_30
        # feature vector: stack sequences (CGM, carbs, bolus, basal)
        feat = np.stack([seq_cgm, seq_carbs, seq_bolus, seq_basal], axis=1)
        Xs.append(feat)
        ys.append(y)
    Xs = np.array(Xs)  # shape (N, LOOKBACK, 4)
    ys = np.array(ys).reshape(-1,1)
    # scale
    nsamples, ntime, nfeat = Xs.shape
    Xflat = Xs.reshape(nsamples, ntime*nfeat)
    scalerX = StandardScaler().fit(Xflat)
    Xs_s = scalerX.transform(Xflat).reshape(nsamples, ntime, nfeat)
    scalerY = StandardScaler().fit(ys)
    ys_s = scalerY.transform(ys)
    # split train/test simple
    split = int(len(Xs_s)*0.8)
    Xtrain, Xtest = Xs_s[:split], Xs_s[split:]
    ytrain, ytest = ys_s[:split], ys_s[split:]
    # model
    model = Sequential([LSTM(32, input_shape=(LOOKBACK, nfeat)), Dense(16, activation='relu'), Dense(1)])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    model.fit(Xtrain, ytrain, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(Xtest,ytest), verbose=0)

    # forecasting: pick a time index near end and forecast STEPS_AHEAD*PRED_STEP minutes ahead
    # pick last available window from data
    idx0 = len(sub)-LOOKBACK-PRED_STEP*STEPS_AHEAD
    if idx0 < LOOKBACK:
        idx0 = len(sub)-LOOKBACK-PRED_STEP
    base_idx = idx0+LOOKBACK
    window = sub.iloc[base_idx-LOOKBACK:base_idx]
    seq = np.stack([window['CGM'].values, window['carbs'].values, window['bolus'].values, window['basal'].values], axis=1)
    seq_scaled = scalerX.transform(seq.reshape(1,LOOKBACK,nfeat).reshape(1,LOOKBACK*nfeat)).reshape(1,LOOKBACK,nfeat)

    preds = []
    berg_state_G = seq[-1,0]
    # for future inputs we'll use zeros for carbs and bolus, last basal
    future_carbs_full = np.zeros(PRED_STEP)
    future_bolus_full = np.zeros(PRED_STEP)
    future_basal_full = np.array([seq[-1,3]]*PRED_STEP)
    cur_seq = seq.copy()
    for step in range(STEPS_AHEAD):
        # berg prediction 30min ahead
        berg30 = bergman_predict(berg_state_G, future_bolus_full, future_carbs_full, future_basal_full, dt=5.0, steps=PRED_STEP)[-1]
        # predict residual with LSTM
        inp = scalerX.transform(cur_seq.reshape(1,LOOKBACK*nfeat)).reshape(1,LOOKBACK,nfeat)
        r_s = model.predict(inp, verbose=0)
        r = scalerY.inverse_transform(r_s.reshape(-1,1))[0,0]
        corrected = berg30 + r
        preds.append(corrected)
        # advance berg_state_G to corrected (as if we observed corrected)
        berg_state_G = corrected
        # update sequence by shifting in predicted cgm and zeros for carbs/bolus and last basal
        new_row = np.array([corrected, 0.0, 0.0, cur_seq[-1,3]])
        cur_seq = np.vstack([cur_seq[1:], new_row])

    # build time axis in hours for plotting: use actual timestamps for the window + future
    times = list(sub['date'].iloc[base_idx-LOOKBACK:base_idx].values)
    start_time = sub['date'].iloc[base_idx]
    future_times = [start_time + pd.Timedelta(minutes=30*(i+1)) for i in range(len(preds))]
    # actual series for plotting: take actual cgm from same start to end of forecast horizon if available
    actual_times = list(sub['date'].iloc[base_idx-LOOKBACK:base_idx+PRED_STEP*STEPS_AHEAD].values)
    actual_cgm = sub['CGM'].iloc[base_idx-LOOKBACK:base_idx+PRED_STEP*STEPS_AHEAD].values

    # plot: only compare actual CGM and predictions in same time scale (hours)
    # create x in hours relative to start_time
    x_actual = [(t - start_time).total_seconds()/3600.0 for t in actual_times]
    x_pred = [(t - start_time).total_seconds()/3600.0 for t in future_times]

    ax = axes[si]
    ax.plot(x_actual, actual_cgm, label='Actual CGM')
    # plot predictions as points at 30-min intervals
    ax.plot(x_pred, preds, marker='o', linestyle='--', label='Hybrid LSTM+Bergman preds')
    ax.set_title(f'Subject id: {sid}')
    ax.set_xlabel('time (hours)')
    ax.set_ylabel('blood glucose (mg/dL)')
    ax.legend()

plt.tight_layout()
plt.show()


# --- Metrics evaluation cell (append to single-cell notebook) ---
# This cell computes glucose-weighted metrics (gRMSE, gMAE), RMSE/MAE/MAPE
# for the hybrid Bergman + LSTM residual-corrector implementation above.
# It rebuilds the training/test split per subject (same logic as the model cell)
# and aggregates final-step predictions across the SUBJECTS_TO_PLOT subjects.
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt
from collections import Counter
import matplotlib.pyplot as plt

# --- Glucose-weighted penalty functions (from your snippet) ---
def sigmoid(x, a, epsilon):
    xi = (2 / epsilon) * (x - a - (epsilon / 2))
    if x <= a:
        return 0
    elif a < x <= a + (epsilon / 2):
        return -0.5 * xi ** 4 - xi ** 3 + xi + 0.5
    elif a + (epsilon / 2) < x <= a + epsilon:
        return 0.5 * xi ** 4 - xi ** 3 + xi + 0.5
    else:
        return 1


def sigmoid_hat(x, a, epsilon):
    xi_hat = - (2 / epsilon) * (x - a + (epsilon / 2))
    if x <= a - epsilon:
        return 1
    elif a - epsilon < x <= a - (epsilon / 2):
        return 0.5 * xi_hat ** 4 - xi_hat ** 3 + xi_hat + 0.5
    elif a - (epsilon / 2) < x <= a:
        return -0.5 * xi_hat ** 4 - xi_hat ** 3 + xi_hat + 0.5
    else:
        return 0


def penalty(g, g_hat):
    alpha_L = 1.5; alpha_H = 1
    beta_L = 30;  beta_H = 100
    gamma_L = 10; gamma_H = 20
    T_L = 85;     T_H = 155
    sigma_T_L = sigmoid_hat(g, T_L, beta_L)
    sigma_gamma_L = sigmoid(g_hat, g, gamma_L)
    sigma_T_H = sigmoid(g, T_H, beta_H)
    sigma_gamma_H = sigmoid_hat(g_hat, g, gamma_H)
    return (1 + alpha_L * sigma_T_L * sigma_gamma_L + alpha_H * sigma_T_H * sigma_gamma_H)


def glucose_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    penalties = np.array([penalty(g, g_hat) for g, g_hat in zip(y_true, y_pred)])
    mse = np.nanmean(((y_true - y_pred) ** 2) * penalties)
    return np.sqrt(mse)


def glucose_mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    penalties = np.array([penalty(g, g_hat) for g, g_hat in zip(y_true, y_pred)])
    mae = np.nanmean(np.abs(y_true - y_pred) * penalties)
    return mae

# --- collect final-step predictions similarly to model cell but aggregated ---
y_true_all = []
y_pred_all = []
per_subject_counts = {}

subjects = df['id'].unique()[:SUBJECTS_TO_PLOT]
for sid in subjects:
    sub = df[df['id']==sid].sort_values('date').reset_index(drop=True)
    cgm = sub['CGM'].values
    carbs = sub['carbs'].fillna(0).values
    bolus = sub['bolus'].fillna(0).values
    basal = sub['basal'].fillna(0).values
    # build dataset
    Xs = []
    ys = []
    berg30s = []
    true_cgms = []
    for i in range(LOOKBACK, len(sub)-PRED_STEP):
        seq_cgm = cgm[i-LOOKBACK:i]
        seq_carbs = carbs[i-LOOKBACK:i]
        seq_bolus = bolus[i-LOOKBACK:i]
        seq_basal = basal[i-LOOKBACK:i]
        future_insulin = bolus[i:i+PRED_STEP].tolist()
        future_carbs = carbs[i:i+PRED_STEP].tolist()
        future_basal = basal[i:i+PRED_STEP].tolist()
        G0 = cgm[i-1]
        berg = bergman_predict(G0, future_insulin, future_carbs, future_basal, dt=5.0, steps=PRED_STEP)
        berg_30 = berg[-1]
        y = cgm[i+PRED_STEP-1] - berg_30
        feat = np.stack([seq_cgm, seq_carbs, seq_bolus, seq_basal], axis=1)
        Xs.append(feat)
        ys.append(y)
        berg30s.append(berg_30)
        true_cgms.append(cgm[i+PRED_STEP-1])
    if len(Xs) == 0:
        per_subject_counts[sid] = 0
        continue
    Xs = np.array(Xs); ys = np.array(ys).reshape(-1,1)
    nsamples, ntime, nfeat = Xs.shape
    Xflat = Xs.reshape(nsamples, ntime*nfeat)
    scalerX = StandardScaler().fit(Xflat)
    Xs_s = scalerX.transform(Xflat).reshape(nsamples, ntime, nfeat)
    scalerY = StandardScaler().fit(ys)
    ys_s = scalerY.transform(ys)
    split = int(len(Xs_s)*0.8)
    Xtrain, Xtest = Xs_s[:split], Xs_s[split:]
    ytrain, ytest = ys_s[:split], ys_s[split:]
    # fit model quickly (same architecture)
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from tensorflow.keras.optimizers import Adam
    model = Sequential([LSTM(32, input_shape=(LOOKBACK, nfeat)), Dense(16, activation='relu'), Dense(1)])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    model.fit(Xtrain, ytrain, epochs=EPOCHS, batch_size=BATCH_SIZE, validation_data=(Xtest,ytest), verbose=0)
    # predict residuals on test set
    r_s = model.predict(Xtest, verbose=0)
    r = scalerY.inverse_transform(r_s.reshape(-1,1)).reshape(-1)
    berg30_test = np.array(berg30s[split:])
    true_test = np.array(true_cgms[split:])
    preds_corrected = berg30_test + r
    # collect
    y_true_all.append(true_test)
    y_pred_all.append(preds_corrected)
    per_subject_counts[sid] = len(true_test)

# flatten
if len(y_true_all) == 0:
    raise RuntimeError("No predictions collected across subjects — run model cell first and ensure enough data.")

y_true = np.concatenate(y_true_all)
y_pred = np.concatenate(y_pred_all)

# Basic numeric metrics
rmse = sqrt(mean_squared_error(y_true, y_pred))
mae  = mean_absolute_error(y_true, y_pred)
nonzero = y_true != 0
mape = np.mean(np.abs((y_pred - y_true)[nonzero] / y_true[nonzero]))*100 if nonzero.any() else np.nan

# glucose-weighted metrics
g_rmse = glucose_rmse(y_true, y_pred)
g_mae  = glucose_mae(y_true, y_pred)

print(f"Hybrid model (final predicted step) — aggregated over subjects:")
print(f"  Samples evaluated : {len(y_true)} (per-subject counts: {per_subject_counts})")
print(f"  RMSE (mg/dL): {rmse:.3f}")
print(f"  MAE  (mg/dL): {mae:.3f}")
print(f"  MAPE (%): {mape:.3f}")
print(f"  Glucose-weighted RMSE (gRMSE, mg/dL): {g_rmse:.3f}")
print(f"  Glucose-weighted MAE  (gMAE, mg/dL): {g_mae:.3f}")

# Simple Clarke & Parkes fallbacks (diagnostic heuristics)

def clarke_zone_for_point(ref, pred):
    if np.isnan(ref) or np.isnan(pred):
        return None
    if (ref >= 70 and abs(pred - ref) <= 0.2 * ref) or (ref < 70 and abs(pred - ref) <= 20):
        return 'A'
    if (pred >= 330 and ref <= 50) or (pred <= 50 and ref >= 330):
        return 'E'
    if ref < 70 and pred >= 180:
        return 'C'
    if ref >= 180 and pred < 70:
        return 'D'
    return 'B'


def parkes_zone_for_point(ref, pred):
    if np.isnan(ref) or np.isnan(pred):
        return None
    err = pred - ref
    rel_err = abs(err) / max(ref, 1.0)
    if (ref < 70 and abs(err) <= 15) or (ref >= 70 and rel_err <= 0.2):
        return 'A'
    if rel_err <= 0.35:
        return 'B'
    if (ref < 70 and pred > 180) or (ref > 180 and pred < 70):
        return 'D'
    if rel_err > 0.6:
        return 'E'
    return 'C'

clarke_zones = Counter([clarke_zone_for_point(r,p) for r,p in zip(y_true, y_pred)])
parkes_zones = Counter([parkes_zone_for_point(r,p) for r,p in zip(y_true, y_pred)])

def zones_to_percent(zdict, total):
    out = {}
    for k, v in zdict.items():
        if k is None:
            continue
        out[str(k)] = (int(v), 100.0 * int(v) / total)
    return out

print('
Clarke error grid (zone : count, %):')
for z, (ct, pct) in zones_to_percent(clarke_zones, len(y_true)).items():
    print(f"  {z}: {ct} ({pct:.2f}%)")

print('
Parkes error grid (zone : count, %):')
for z, (ct, pct) in zones_to_percent(parkes_zones, len(y_true)).items():
    print(f"  {z}: {ct} ({pct:.2f}%)")

# Glycemia confusion matrix (Hypo <70 ; Normo 70-180 ; Hyper >180)
HYPO = 70
HYPER = 180

def glycemia_class(x, hypo=HYPO, hyper=HYPER):
    x = np.asarray(x)
    cls = np.full(x.shape, -1, dtype=int)
    cls[x < hypo] = 0
    cls[(x >= hypo) & (x <= hyper)] = 1
    cls[x > hyper] = 2
    return cls


def compute_glycemia_confusion(y_true, y_pred, hypo=HYPO, hyper=HYPER):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]; y_pred = y_pred[mask]
    counts = np.zeros((3,3), dtype=int)
    if y_true.size == 0:
        return counts, np.full_like(counts, np.nan, dtype=float)
    true_cls = glycemia_class(y_true, hypo=hypo, hyper=hyper)
    pred_cls = glycemia_class(y_pred, hypo=hypo, hyper=hyper)
    for t, p in zip(true_cls, pred_cls):
        if 0 <= t <= 2 and 0 <= p <= 2:
            counts[t, p] += 1
    pct = np.full_like(counts, np.nan, dtype=float)
    for i in range(3):
        row_sum = counts[i].sum()
        if row_sum > 0:
            pct[i, :] = counts[i, :] / float(row_sum)
        else:
            pct[i, :] = np.nan
    return counts, pct

counts, pct = compute_glycemia_confusion(y_true, y_pred)
print('
Glycemia detection — counts (rows=true, cols=pred):
', counts)
print('
Glycemia detection — row-normalized % (rows sum to 100%):
', np.round(pct * 100, 2))

# Simple scatter + confusion heatmap
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax0 = ax[0]
ax0.scatter(y_true, y_pred, s=8, alpha=0.5)
ax0.plot([0, 400], [0, 400], '--', linewidth=0.8)
xs = np.linspace(0, 400, 100)
ax0.plot(xs, 1.2 * xs, linewidth=0.6)
ax0.plot(xs, 0.8 * xs, linewidth=0.6)
ax0.axvline(70, linestyle=':', linewidth=0.6); ax0.axhline(70, linestyle=':', linewidth=0.6)
ax0.set_xlim(0, 400); ax0.set_ylim(0, 400)
ax0.set_xlabel('Reference CGM (mg/dL)'); ax0.set_ylabel('Predicted CGM (mg/dL)')
ax0.set_title('Predicted vs Reference (Clarke guidance lines)')

ax1 = ax[1]
im = ax1.imshow(pct, vmin=0.0, vmax=1.0, cmap='Blues', interpolation='nearest')
ax1.set_xticks(np.arange(3)); ax1.set_yticks(np.arange(3))
labels = ['Pred_Hypo
(<70)', 'Pred_Normo
(70-180)', 'Pred_Hyper
(>180)']
ax1.set_xticklabels(labels); ax1.set_yticklabels(['True_Hypo','True_Normo','True_Hyper'])
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
ax1.set_title('Glycemia detection confusion (row-normalized)')
for i in range(3):
    for j in range(3):
        c = counts[i, j]
        p = pct[i, j]
        if np.isnan(p):
            txt = "n/a
(0)"
        else:
            txt = f"{p*100:4.1f}%
({c})"
        ax1.text(j, i, txt, ha='center', va='center', color='black', fontsize=10)
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

print('
Done. Notes:')
print('- Metrics computed on the final predicted step (same horizon as plotting cell).')
print('- The glucose-weighted metrics (gRMSE/gMAE) use the penalty functions provided.')


In [ ]:
# --- Metrics evaluation cell (append to single-cell notebook) ---
# This cell computes glucose-weighted metrics (gRMSE, gMAE), RMSE/MAE/MAPE
# for the hybrid Bergman + LSTM residual-corrector implementation above.
# It rebuilds the training/test split per subject (same logic as the model cell)
# and aggregates final-step predictions across the SUBJECTS_TO_PLOT subjects.
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt
from collections import Counter
import matplotlib.pyplot as plt


# --- Glucose-weighted penalty functions (from your snippet) ---
def sigmoid(x, a, epsilon):
xi = (2 / epsilon) * (x - a - (epsilon / 2))
if x <= a:
return 0
elif a < x <= a + (epsilon / 2):
return -0.5 * xi ** 4 - xi ** 3 + xi + 0.5
elif a + (epsilon / 2) < x <= a + epsilon:
return 0.5 * xi ** 4 - xi ** 3 + xi + 0.5
else:
return 1




def sigmoid_hat(x, a, epsilon):
xi_hat = - (2 / epsilon) * (x - a + (epsilon / 2))
if x <= a - epsilon:
return 1
elif a - epsilon < x <= a - (epsilon / 2):
return 0.5 * xi_hat ** 4 - xi_hat ** 3 + xi_hat + 0.5
elif a - (epsilon / 2) < x <= a:
return -0.5 * xi_hat ** 4 - xi_hat ** 3 + xi_hat + 0.5
else:
return 0




def penalty(g, g_hat):
alpha_L = 1.5; alpha_H = 1
beta_L = 30; beta_H = 100
gamma_L = 10; gamma_H = 20
T_L = 85; T_H = 155
sigma_T_L = sigmoid_hat(g, T_L, beta_L)
sigma_gamma_L = sigmoid(g_hat, g, gamma_L)
sigma_T_H = sigmoid(g, T_H, beta_H)
sigma_gamma_H = sigmoid_hat(g_hat, g, gamma_H)
return (1 + alpha_L * sigma_T_L * sigma_gamma_L + alpha_H * sigma_T_H * sigma_gamma_H)




def glucose_rmse(y_true, y_pred):
y_true = np.asarray(y_true, dtype=float)
y_pred = np.asarray(y_pred, dtype=float)
mask = np.isfinite(y_true) & np.isfinite(y_pred)
y_true, y_pred = y_true[mask], y_pred[mask]
penalties = np.array([penalty(g, g_hat) for g, g_hat in zip(y_true, y_pred)])
mse = np.nanmean(((y_true - y_pred) ** 2) * penalties)
return np.sqrt(mse)




def glucose_mae(y_true, y_pred):
y_true = np.asarray(y_true, dtype=float)